In [36]:
from dataloader import Dataloader as MyLoader
from models.CNN import SoundCNN
from trainer import Train
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
from adversarial import deepfool, analyze_perturbation
import numpy as np
import matplotlib.pyplot as plt
import librosa
import os
from torch.optim import Adam
from torch.nn import CrossEntropyLoss


In [ ]:
TRAIN_FOLDS = ["fold1"] #[f"fold{i}" for i in range(1, 10)]
TEST_FOLDS = ["fold10"]
DATA_PATH = "datasets"

### TODO

Ultima celula nao funciona. nao sei o que fazer o miguel tem de pegar nisto para garantir que os resultados sao bem guardados e visualizaveis.

O codigo esta com train_folds = [fold1] e epochs = 1. apaguem o train_fold e descomentem o codigo correto e mudem as 3 epochs para 50.

In [38]:
def run_epoch(dataloader, model_instance=None, optimizer=None, criterion=None, device=torch.device("cuda" if torch.cuda.is_available() else "cpu")):
    model_instance.to(device)
    model_instance.train()
    running_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(dataloader, desc="Training", leave=True)
    for batch_idx, (inputs, folds, labels) in enumerate(progress_bar):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_instance(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        current_acc = correct / total if total > 0 else 0.0
        current_loss = running_loss / total if total > 0 else 0.0
        progress_bar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})
    avg_loss = running_loss / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
    return avg_loss, accuracy

In [39]:
model_instance = SoundCNN(num_classes=10, in_channels=5)
train_loader = DataLoader(MyLoader(dataset_path=DATA_PATH, folds=TRAIN_FOLDS, include_augmented=False, use_cache=True), batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

patience = 5
best_loss = float('inf')
epochs_no_improve = 0
criterion = CrossEntropyLoss()
optimizer = Adam(model_instance.parameters(), lr=0.001, weight_decay=1e-5)

for epoch in range(1, 2):
        print(f"\n--- Epoch {epoch}/50 ---")
        training_loss, training_acc = run_epoch(train_loader, model_instance=model_instance, device=device, optimizer=optimizer, criterion=criterion)

        print(f"Training Loss: {training_loss:.4f}, Training Accuracy: {training_acc:.4f}")

        # Early Stopping Check
        if training_loss < best_loss:
            best_loss = training_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break


torch.save(model_instance.state_dict(), os.path.join(os.getcwd(), "adversarial_results", "CNN", "model.pth"))
     

Using device: cuda

--- Epoch 1/50 ---


Training: 100%|██████████| 14/14 [00:01<00:00,  9.89it/s, loss=2.0069, acc=0.3368]

Training Loss: 2.0069, Training Accuracy: 0.3368


In [40]:
model_instance = SoundCNN(num_classes=10, AttentionBlock=True, in_channels=5)
train_loader = DataLoader(MyLoader(dataset_path=DATA_PATH, folds=TRAIN_FOLDS, include_augmented=False, use_cache=True), batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

patience = 5
best_loss = float('inf')
epochs_no_improve = 0

criterion = CrossEntropyLoss()
optimizer = Adam(model_instance.parameters(), lr=0.001, weight_decay=1e-5)

for epoch in range(1, 2):
        print(f"\n--- Epoch {epoch}/50 ---")
        training_loss, training_acc = run_epoch(train_loader, model_instance=model_instance, device=device, optimizer=optimizer, criterion=criterion)

        print(f"Training Loss: {training_loss:.4f}, Training Accuracy: {training_acc:.4f}")

        # Early Stopping Check
        if training_loss < best_loss:
            best_loss = training_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break


torch.save(model_instance.state_dict(), os.path.join(os.getcwd(), "adversarial_results", "ACNN", "model.pth"))
     

Using device: cuda

--- Epoch 1/50 ---


Training: 100%|██████████| 14/14 [00:01<00:00, 10.77it/s, loss=2.0183, acc=0.3288]

Training Loss: 2.0183, Training Accuracy: 0.3288


In [41]:
model_instance = SoundCNN(num_classes=10, AttentionBlock=True, SqueezeExcitation=True, in_channels=5)
train_loader = DataLoader(MyLoader(dataset_path=DATA_PATH, folds=TRAIN_FOLDS, include_augmented=False, use_cache=True), batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

patience = 5
best_loss = float('inf')
epochs_no_improve = 0

criterion = CrossEntropyLoss()
optimizer = Adam(model_instance.parameters(), lr=0.001, weight_decay=1e-5)
for epoch in range(1, 2):
        print(f"\n--- Epoch {epoch}/50 ---")
        training_loss, training_acc = run_epoch(train_loader, model_instance=model_instance, device=device, optimizer=optimizer, criterion=criterion)

        print(f"Training Loss: {training_loss:.4f}, Training Accuracy: {training_acc:.4f}")

        # Early Stopping Check
        if training_loss < best_loss:
            best_loss = training_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break


torch.save(model_instance.state_dict(), os.path.join(os.getcwd(), "adversarial_results", "ASECNN", "model.pth"))
     

Using device: cuda

--- Epoch 1/50 ---


Training: 100%|██████████| 14/14 [00:01<00:00, 13.49it/s, loss=2.1405, acc=0.2955]

Training Loss: 2.1405, Training Accuracy: 0.2955


In [42]:
class_mapping = {
            0: 'air_conditioner',
            1: 'car_horn',
            2: 'children_playing',
            3: 'dog_bark',
            4: 'drilling',
            5: 'engine_idling',
            6: 'gun_shot',
            7: 'jackhammer',
            8: 'siren',
            9: 'street_music'
        }

In [43]:

def plot_all_channels(model_name, features : np.ndarray, fold : int = None, label : str = None):
    
    title = "Audio Features"
    
    if fold is not None and label is not None:
        title += f" (Label: {label}, Fold: {fold})"
    elif fold is not None:
        title += f" (Fold: {fold})"
    elif label is not None:
        title += f" (Label: {label})"
    
    titles = ['Log Mel Spectrogram', 'Delta', 'Delta-Delta', 'Harmonic', 'Percussive']
    fig, axes = plt.subplots(5, 1, figsize=(10, 15))
    fig.suptitle(title)
    
    for i, ax in enumerate(axes):
        img = librosa.display.specshow(features[i], x_axis='time', y_axis='mel', sr=22050, ax=ax)
        ax.set_title(titles[i])
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        
    plt.tight_layout()
    #plt.show()
    plt.savefig(f"adversarial_results/{model_name}/adversarial_example_label_{label}.png")
    plt.close()


In [48]:
import uuid

models = ["CNN", "ACNN", "ASECNN"]
for model_name in models:
    test_loader = DataLoader(
        MyLoader(dataset_path=DATA_PATH, folds=TEST_FOLDS, include_augmented=False, use_cache=True),
        batch_size=64, shuffle=False
    )
    path = os.path.join(os.getcwd(), "adversarial_results", model_name, "model.pth")
    state = torch.load(path)
    model = SoundCNN(
        num_classes=10,
        AttentionBlock=("A" in model_name),
        SqueezeExcitation=("SE" in model_name),
        in_channels=5
    )
    model.load_state_dict(state)
    model.to(device)
    model.eval()

    save_dir = os.path.join(os.getcwd(), "adversarial_results", model_name)
    os.makedirs(save_dir, exist_ok=True)

    for batch_idx, (inputs, folds, labels) in enumerate(test_loader):
        inputs, labels, folds = inputs.to(device), labels.to(device), folds.to(device) if torch.is_tensor(folds) else folds

        for i in range(inputs.size(0)):
            input_i = inputs[i].unsqueeze(0)
            label_i = labels[i].item()
            fold_i = folds[i].item() if torch.is_tensor(folds) else folds[i]
            original_label = class_mapping[label_i]

            noise, adv_img, success = deepfool(model, input_i, max_iter=20)

            # Gera um nome único para cada ficheiro
            file_id = str(uuid.uuid4())
            base_name = f"fold{fold_i}_idx{batch_idx}_{i}_{original_label}"

            if success:
                with torch.no_grad():
                    new_out = model(adv_img)
                    new_pred = torch.argmax(new_out, dim=1).item()
                    new_label = class_mapping[new_pred]
                print(f"SUCCESS! Model fooled: {original_label} -> {new_label}")
                worst_channel = analyze_perturbation(noise)
                print(f"   Most attacked channel: {worst_channel}")

                adv_numpy = adv_img.squeeze(0).cpu().detach().numpy()
                noise_numpy = noise.squeeze(0).cpu().detach().numpy()
                features = input_i.squeeze(0).cpu().detach().numpy()

                # Salva os resultados sem overwrite
                np.save(os.path.join(save_dir, f"{base_name}_original_{file_id}.npy"), features)
                np.save(os.path.join(save_dir, f"{base_name}_adversarial_{file_id}.npy"), adv_numpy)
                np.save(os.path.join(save_dir, f"{base_name}_noise_{file_id}.npy"), noise_numpy)

                # Salva as imagens
                plot_all_channels(model_name, features, fold_i, f"Original: {original_label}")
                os.rename(
                    f"adversarial_results/{model_name}/adversarial_example_label_Original{original_label}.png",
                    os.path.join(save_dir, f"{base_name}_original_{file_id}.png")
                )
                plot_all_channels(model_name, adv_numpy, fold_i, f"Adversarial {new_label}")
                os.rename(
                    f"adversarial_results/{model_name}/adversarial_example_label_Adversarial{new_label}.png",
                    os.path.join(save_dir, f"{base_name}_adversarial_{file_id}.png")
                )
                plot_all_channels(model_name, noise_numpy * 100, fold_i, f"Pure Noise (x100) - Target {new_label}")
                os.rename(
                    f"adversarial_results/{model_name}/adversarial_example_label_Pure Noise (x100) - Target {new_label}.png",
                    os.path.join(save_dir, f"{base_name}_noise_{file_id}.png")
                )
            else:
                print("Failed to fool the model (Example might be too robust or max_iter too low).")

SUCCESS! Model fooled: dog_bark -> siren
   [Noise Analysis] Energy per channel:
     - Log Mel: 37.8528
     - Delta: 33.0965
     - Delta-Delta: 31.6100
     - Harmonic: 39.9439
     - Percussive: 40.9218
   Most attacked channel: Percussive


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'adversarial_results/CNN/adversarial_example_label_Originaldog_bark.png' -> 'c:\\Users\\diogo\\OneDrive\\Documents\\GitHub\\deep-learning-urban-sound-data\\adversarial_results\\CNN\\fold10_idx0_0_dog_bark_original_3cceb562-507c-43ec-bdd3-e9aaef0bf6b7.png'